# a2a-test · ProteinMAN (IA y Data Science aplicada a bioinformática) · segundo agente

**Autor del agente:** Fabio Durán · Grupo ⟨GRUPO⟩ (Fabio Durán · Gustavo Reyes · Ezequiel Muñoz)

> **Nota de reconstrucción.** El notebook `a2a-test.ipynb` de la clase no estaba disponible. Este notebook
> se reconstruyó con el código que trae el enunciado del Taller 05 (celdas copiadas al pie de la letra) y,
> para los bloques 2 a 5 de «Ejemplo real», con el supervisor descrito en la clase 02 (Decision con
> salida estructurada y la línea `[Supervisor]`).

Kernel: el Python de `.venv` de este proyecto. Servidor: `langgraph dev --port 2026` (grafo `proteinman`).

## Configuración
Primera celda: conexión al puerto del agente.

In [1]:
LANGGRAPH_URL = "http://127.0.0.1:2026"
A2A_HOST = "127.0.0.1"
A2A_PORT = 2026
A2A_URL = f"http://{A2A_HOST}:{A2A_PORT}"

import json
from langgraph_sdk import get_client

lg = get_client(url=LANGGRAPH_URL, api_key=None)


def mostrar(obj):
    """Imprime un dict/lista como JSON legible."""
    print(json.dumps(obj, indent=2, ensure_ascii=False, default=str))

import sys
print("Kernel:", sys.executable)
print("Conectado a", LANGGRAPH_URL)

Kernel: /home/fabio/repositorio/curso-agentes/.venv/bin/python
Conectado a http://127.0.0.1:2026


## Encontrar el Id de tu agente
El servidor asigna un UUID (`assistant_id`) a cada grafo. Se busca el grafo elegido y se guarda su UUID en `ASSISTANT_ID`.

In [2]:
asistentes = await lg.assistants.search(limit=100, response_format="array")
for a in asistentes:
    print(f"{a['graph_id']:<18} → {a['assistant_id']}  ({a['name']})")

GRAFO = "proteinman"
ASSISTANT_ID = next(a["assistant_id"] for a in asistentes if a["graph_id"] == GRAFO)
print("\nASSISTANT_ID =", ASSISTANT_ID)

proteinman         → 988d2098-af3e-5d3f-9bec-7ace332b7936  (ProteinMAN)

ASSISTANT_ID = 988d2098-af3e-5d3f-9bec-7ace332b7936


## Funciones de apoyo: `crear_entrada`, `extraer_texto`, `consultar_langgraph`

In [3]:
def crear_entrada(texto: str) -> dict:
    """Arma la entrada del grafo: el estado tiene la clave `messages`."""
    return {"messages": [{"role": "user", "content": texto}]}


def _texto_de(contenido) -> str:
    """El contenido puede ser un str o una lista de bloques (Gemini devuelve bloques {'type': 'text'})."""
    if isinstance(contenido, str):
        return contenido
    return "".join(b.get("text", "") if isinstance(b, dict) else str(b) for b in contenido)


def extraer_texto(estado: dict) -> str:
    """Devuelve el texto del último mensaje del agente (tipo 'ai')."""
    for m in reversed(estado.get("messages", [])):
        if m.get("type") == "ai" and m.get("content"):
            return _texto_de(m["content"])
    return "(sin respuesta de texto)"


def herramientas_usadas(estado: dict) -> list[str]:
    """Nombres de las herramientas que el agente llamó en esta ejecución (evidencia de uso de los apuntes)."""
    return [m.get("name") for m in estado.get("messages", []) if m.get("type") == "tool"]


async def consultar_langgraph(texto: str, thread_id: str | None = None) -> dict:
    """Ejecuta el grafo y espera el estado final. Con thread_id, conserva la memoria de esa conversación."""
    return await lg.runs.wait(thread_id, ASSISTANT_ID, input=crear_entrada(texto))

print("Funciones listas.")

Funciones listas.


## Comprobación rápida del agente

In [4]:
# Cambia esta pregunta por una de tu especialidad.
PREGUNTA = "Hola, ¿quién eres y en qué me puedes ayudar?"

estado = await lg.runs.wait(
    None,
    ASSISTANT_ID,
    input=crear_entrada(PREGUNTA),
)
print(extraer_texto(estado))
print("\n[herramientas usadas]", herramientas_usadas(estado))

Soy ProteinMAN, especialista en IA y Data Science aplicada a bioinformática, creado por Fabio Durán. Te ayudo a elegir técnicas de machine learning, métricas de evaluación, modelos de arranque según tus datos biológicos y a consultar un catálogo de modelos con ejemplos en Python.

[herramientas usadas] []


## Tarjeta por A2A

In [5]:
import httpx
from a2a.client import A2ACardResolver
from google.protobuf.json_format import MessageToDict

http_a2a = httpx.AsyncClient(timeout=120, trust_env=False)
resolver = A2ACardResolver(http_a2a, A2A_URL)
card_remota = await resolver.get_agent_card(
    http_kwargs={"params": {"assistant_id": ASSISTANT_ID}}
)
mostrar(MessageToDict(card_remota))

{
  "name": "ProteinMAN",
  "description": "Especialista en IA y Data Science aplicada a bioinformática. Puede: recomendar técnicas de machine learning para problemas con datos biológicos, sugerir métricas para evaluar modelos bioinformáticos, elegir un modelo según el tipo de dato biológico (secuencias, expresión génica, estructuras).",
  "supportedInterfaces": [
    {
      "url": "http://127.0.0.1:2026/a2a/988d2098-af3e-5d3f-9bec-7ace332b7936",
      "protocolBinding": "JSONRPC",
      "protocolVersion": "1.0"
    }
  ],
  "version": "0.17.1",
  "capabilities": {
    "streaming": true,
    "pushNotifications": false,
    "extensions": [
      {
        "uri": "https://langchain.com/a2a/extensions/history-scope/v1",
        "description": "Choose task-only or full-context response history",
        "params": {
          "header": "LangGraph-A2A-History-Scope",
          "methods": [
            "SendMessage",
            "GetTask",
            "ListTasks"
          ],
          "valu

## Preparar la tarjeta para que el supervisor reconozca al agente

La celda es la del enunciado con un solo cambio: `name=nombre`, porque este agente se presenta como **ProteinMAN** y no como «Agente …».

In [6]:
# Completa los mismos datos que elegiste para tu prompt.
nombre = "ProteinMAN"
especialidad = "IA y Data Science aplicada a bioinformática"
skills = "recomendar técnicas de machine learning para problemas con datos biológicos, sugerir métricas para evaluar modelos bioinformáticos, elegir un modelo según el tipo de dato biológico (secuencias, expresión génica, estructuras)"

await lg.assistants.update(
    ASSISTANT_ID,
    name=nombre,
    description=f"Especialista en {especialidad}. Puede: {skills}.",
)
print("Tarjeta actualizada con mi especialidad y mis habilidades.")

Tarjeta actualizada con mi especialidad y mis habilidades.


Comprobación: la tarjeta que entrega el servidor ya trae el nombre y la descripción nuevos.

In [7]:
card_actualizada = await resolver.get_agent_card(
    http_kwargs={"params": {"assistant_id": ASSISTANT_ID}}
)
print("name:       ", card_actualizada.name)
print("description:", card_actualizada.description)
print("skills:     ", [s.name for s in card_actualizada.skills])

name:        ProteinMAN
description: Especialista en IA y Data Science aplicada a bioinformática. Puede: recomendar técnicas de machine learning para problemas con datos biológicos, sugerir métricas para evaluar modelos bioinformáticos, elegir un modelo según el tipo de dato biológico (secuencias, expresión génica, estructuras).
skills:      ['ProteinMAN Capabilities']


In [8]:
await http_a2a.aclose()
print("Conexiones cerradas.")

Conexiones cerradas.
